# Day 31 — Power Transformer: Box–Cox and Yeo–Johnson

**Course:** CampusX — 100 Days of Machine Learning  
**Main idea:** Learn how power transformations change the shape of numerical features and how to apply them safely before training a model.

In this notebook, we will:

1. Inspect the concrete-strength dataset.
2. Train a baseline Linear Regression model on the original features.
3. Examine feature distributions using histograms and Q–Q plots.
4. Apply Box–Cox and Yeo–Johnson transformations.
5. Compare the transformed features and model scores.
6. Use a Pipeline and cross-validation so preprocessing is fitted separately inside each training fold.

> **How to use this notebook:** Run the cells from top to bottom. The explanations here focus on what we are doing, why we are doing it, and what to observe. The detailed theory and mathematical derivations are in `notes.md`.


## 1. Understanding the problem: why transform numerical features?

A numerical feature can contain many small or medium values and a few extremely large values. This creates a **right-skewed distribution**: most observations are concentrated on the left, while a long tail extends to the right.

For example:

```python
values = [10, 12, 15, 18, 20, 25, 30, 500, 800, 1000]
```

A few large values stretch the scale. Some algorithms can be sensitive to this shape or to differences in feature magnitude. A power transformation changes the spacing between values so that a long tail may become less pronounced.

**Important:** the transformation does not delete observations or guarantee a perfectly normal distribution. We will inspect the distributions and compare model performance rather than assume the transformation helped.

### What we are comparing

- **Original features:** no power transformation.
- **Box–Cox:** estimates a power parameter for each feature; input values must be strictly positive.
- **Yeo–Johnson:** also estimates a power parameter, but supports positive, zero, and negative values.


## 2. Import the libraries

We need Pandas to work with the dataset, Matplotlib and Seaborn to plot distributions, SciPy to create Q–Q plots, and Scikit-learn for splitting data, transforming features, training models, and evaluating them.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PowerTransformer, FunctionTransformer
from sklearn.pipeline import Pipeline


## 3. Load and inspect the concrete-strength dataset

The dataset contains concrete ingredients as input features and compressive strength as the target. We first load the CSV and inspect a few rows so we know what the columns look like.

The relative path below assumes that this notebook is inside the day folder and the CSV is inside its `datasets/` folder.

In [ ]:
# Load the dataset from the datasets folder.
df = pd.read_csv("datasets/concrete_data.csv")

# Display the first rows to inspect the column names and sample values.
df.head()


### Check missing values

`PowerTransformer` expects numerical, finite input. This check tells us whether any columns contain missing values before preprocessing.

In [ ]:
# Count missing values in each column.
df.isnull().sum()


## 4. Separate features (X) and target (y)

We want to predict the concrete's compressive strength.

- `X` contains the input features used by the model.
- `y` contains the target value the model must predict.

We select `Strength` by its column name so the code does not depend on it being the last column.

In [ ]:
# Keep all ingredient and age columns as input features.
X = df.drop(columns=["Strength"])

# The value we want the regression model to predict.
y = df["Strength"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


## 5. Split the data into training and testing sets

We keep 20% of the observations for testing. The model learns from `X_train` and `y_train`; the test set is held aside to evaluate how well it predicts observations it did not train on.

We set `random_state=34` so the split is reproducible when the same dataset and library behavior are used.

In [ ]:
# Split the observations before learning any preprocessing parameters.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=34
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)


## 6. Train a baseline Linear Regression model

Before transforming anything, we need a reference score. This is called the **baseline**.

We train Linear Regression using the original features, predict the test targets, and calculate the R² score. Later, we will compare this score with the scores obtained after transformation.

In [ ]:
# Create and train the baseline model on the original training features.
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict the target values for observations kept aside for testing.
y_pred = lr.predict(X_test)

# Compare predictions with the actual test targets.
baseline_r2 = r2_score(y_test, y_pred)
print("Baseline test R²:", baseline_r2)


### Baseline cross-validation score

A single train/test split gives one evaluation result. Cross-validation repeats the train/validation process across several folds and gives us a broader view of model performance.

Here we evaluate the **untransformed baseline** using five folds. We will compare this with complete preprocessing Pipelines later.

In [ ]:
# Each fold trains a fresh Linear Regression model on its training portion.
baseline_cv_scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=10,
    scoring="r2"
)

print("Baseline CV scores:", baseline_cv_scores)
print("Mean baseline CV R²:", baseline_cv_scores.mean())


## 7. Inspect feature distributions before transformation

We inspect each training feature using two plots:

- **Histogram:** shows how frequently values occur and whether the distribution has a long tail.
- **Normal Q–Q plot:** compares the feature's sample quantiles with theoretical quantiles from a normal distribution. Points close to the reference line are more consistent with a normal shape.

These plots help us decide which features may be skewed. They are diagnostic tools; they do not prove that a model's assumptions are satisfied.

In [ ]:
# Plot a histogram and a normal Q–Q plot for each training feature.
for col in X_train.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # The histogram shows how frequently different feature values occur.
    axes[0].hist(X_train[col].dropna(), bins=30, density=True, alpha=0.7)
    axes[0].set_title(f"{col} — histogram before transformation")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")

    # The Q–Q plot compares sample quantiles with normal-distribution quantiles.
    stats.probplot(X_train[col].dropna(), dist="norm", plot=axes[1])
    axes[1].set_title(f"{col} — normal Q–Q plot")

    plt.tight_layout()
    plt.show()


## 8. Box–Cox transformation

Box–Cox estimates a separate power parameter (λ) for each feature. It can be used only when every input value is **strictly greater than zero**.

The original implementation adds `0.0001` before fitting. This can move zero values above zero, but it does **not** fix negative values. We therefore inspect the training data first.

> **Important:** adding a small constant is a data adjustment, not a general solution for negative values. If a feature contains negatives, use a transformation that supports them (such as Yeo–Johnson) or justify a suitable shift based on the data.


In [ ]:
# Count values that are zero or negative in each training feature.
non_positive_counts = (X_train <= 0).sum()
display(non_positive_counts)

# A small shift can handle zeros, but it cannot make negative values positive.
if (X_train < 0).any().any() or (X_test < 0).any().any():
    raise ValueError(
        "Box-Cox requires strictly positive values. A shift of 0.0001 "
        "does not fix negative values; use Yeo-Johnson or justify a larger shift."
    )

# The original course example uses this small shift to move zero values above zero.
epsilon = 0.0001
X_train_boxcox_input = X_train + epsilon
X_test_boxcox_input = X_test + epsilon


### Fit Box–Cox on training data and transform both sets

`fit_transform()` estimates the λ values from the training features and transforms those features. `transform()` applies the already-learned λ values to the test features; it does not estimate new values from the test set.

We convert the results back into DataFrames so the original feature names and row indexes remain available for inspection and plotting.

In [ ]:
# Create a PowerTransformer that uses the Box-Cox formula family.
pt_boxcox = PowerTransformer(method="box-cox")

# Learn one lambda per feature using only the training data, then transform it.
X_train_boxcox = pd.DataFrame(
    pt_boxcox.fit_transform(X_train_boxcox_input),
    columns=X_train.columns,
    index=X_train.index
)

# Apply the training-set lambdas to the test data; do not fit again.
X_test_boxcox = pd.DataFrame(
    pt_boxcox.transform(X_test_boxcox_input),
    columns=X_test.columns,
    index=X_test.index
)

# Show the lambda estimated for each input feature.
boxcox_lambdas = pd.DataFrame({
    "feature": X_train.columns,
    "box_cox_lambda": pt_boxcox.lambdas_
})

display(boxcox_lambdas)


### Train and evaluate Linear Regression after Box–Cox

Now we train a new Linear Regression model using the transformed training features. We then transform the test features using the same fitted transformer and evaluate the predictions.

Compare this test R² with the baseline test R². A higher score on one split is useful evidence, but it is not enough by itself to prove that the transformation will generalize better.

In [ ]:
# Train a separate model using the Box-Cox-transformed training features.
lr_boxcox = LinearRegression()
lr_boxcox.fit(X_train_boxcox, y_train)

# Predict with test features transformed using the training-set lambdas.
y_pred_boxcox = lr_boxcox.predict(X_test_boxcox)

# Evaluate the Box-Cox experiment on the same test targets as the baseline.
boxcox_test_r2 = r2_score(y_test, y_pred_boxcox)
print("Box-Cox test R²:", boxcox_test_r2)
print("Baseline test R²:", baseline_r2)


### Compare the distributions before and after Box–Cox

The left plot shows the original training feature. The right plot shows the same feature after Box–Cox and the transformer's default standardization.

Look for changes in skewness and the tail. A more symmetric histogram is evidence that the feature distribution changed; it does not automatically mean that the model will perform better.

In [ ]:
# Compare each original training feature with its Box-Cox-transformed version.
for col in X_train.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(X_train[col], bins=30, density=True, alpha=0.7)
    axes[0].set_title(f"{col} — before Box-Cox")

    axes[1].hist(X_train_boxcox[col], bins=30, density=True, alpha=0.7)
    axes[1].set_title(f"{col} — after Box-Cox")

    axes[0].set_ylabel("Density")
    axes[1].set_ylabel("Density")

    plt.tight_layout()
    plt.show()


### Evaluate Box–Cox with cross-validation

For a fair cross-validation comparison, the transformation must be fitted separately inside each training fold. We therefore put the shift, Box–Cox transformer, and regression model into one Pipeline.

This prevents a validation fold from influencing the λ values used to transform it. The small shift is included only because the course demonstration uses it to handle zero values; the earlier check ensures that the data do not contain negative values.

In [ ]:
# Apply the same fixed small shift inside each fold before Box-Cox.
def add_boxcox_shift(data):
    return data + epsilon

boxcox_pipeline = Pipeline([
    ("shift", FunctionTransformer(add_boxcox_shift, validate=False)),
    ("power", PowerTransformer(method="box-cox")),
    ("model", LinearRegression())
])

# Cross-validation fits the complete pipeline independently in every fold.
boxcox_cv_scores = cross_val_score(
    boxcox_pipeline,
    X,
    y,
    cv=10,
    scoring="r2"
)

print("Box-Cox CV scores:", boxcox_cv_scores)
print("Mean Box-Cox CV R²:", boxcox_cv_scores.mean())


## 9. Yeo–Johnson transformation

Yeo–Johnson uses a related power-transformation family, but unlike Box–Cox it supports **negative, zero, and positive** input values.

This makes it useful when a numerical feature contains zero or negative observations and we do not want to introduce an arbitrary shift just to use Box–Cox.

In Scikit-learn, `PowerTransformer()` uses Yeo–Johnson by default. We will specify the method explicitly so the code clearly communicates which transformation we are testing.

### Fit Yeo–Johnson on training data

We follow the same train/test rule as before:

- Fit the transformer using `X_train`.
- Transform `X_train` with the learned parameters.
- Transform `X_test` using those same parameters.

We use separate variable names for Yeo–Johnson so the Box–Cox results remain available for comparison.

In [ ]:
# Create a PowerTransformer that explicitly uses the Yeo-Johnson method.
pt_yeojohnson = PowerTransformer(method="yeo-johnson")

# Estimate one lambda per feature using only the training data.
X_train_yeojohnson = pd.DataFrame(
    pt_yeojohnson.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Apply the learned Yeo-Johnson transformation to the test features.
X_test_yeojohnson = pd.DataFrame(
    pt_yeojohnson.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Display the fitted lambda value for each feature.
yeojohnson_lambdas = pd.DataFrame({
    "feature": X_train.columns,
    "yeo_johnson_lambda": pt_yeojohnson.lambdas_
})

display(yeojohnson_lambdas)


### Train and evaluate Linear Regression after Yeo–Johnson

We train another Linear Regression model on the Yeo–Johnson-transformed features and evaluate it on the transformed test set.

The score variable is named `yeojohnson_test_r2` so it cannot accidentally display the baseline or Box–Cox score.

In [ ]:
# Train Linear Regression using the Yeo-Johnson-transformed training features.
lr_yeojohnson = LinearRegression()
lr_yeojohnson.fit(X_train_yeojohnson, y_train)

# Predict the target using the transformed test features.
y_pred_yeojohnson = lr_yeojohnson.predict(X_test_yeojohnson)

# Calculate and display the Yeo-Johnson test R² score.
yeojohnson_test_r2 = r2_score(y_test, y_pred_yeojohnson)
print("Yeo-Johnson test R²:", yeojohnson_test_r2)
print("Baseline test R²:", baseline_r2)


### Compare the distributions before and after Yeo–Johnson

This plot uses the Yeo–Johnson results (not the Box–Cox results). Compare the shape of each pair of histograms and notice whether the long tail has become less pronounced.

In [ ]:
# Compare each original training feature with its Yeo-Johnson-transformed version.
for col in X_train.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(X_train[col], bins=30, density=True, alpha=0.7)
    axes[0].set_title(f"{col} — before Yeo-Johnson")

    axes[1].hist(X_train_yeojohnson[col], bins=30, density=True, alpha=0.7)
    axes[1].set_title(f"{col} — after Yeo-Johnson")

    axes[0].set_ylabel("Density")
    axes[1].set_ylabel("Density")

    plt.tight_layout()
    plt.show()


## 10. Use Yeo–Johnson inside a Pipeline

The manual train/test implementation helped us see each step. In a real ML workflow, it is safer to combine preprocessing and model training in a `Pipeline`.

The Pipeline below contains two steps:

1. `power`: fit a Yeo–Johnson transformer and apply it to the features.
2. `lr`: train Linear Regression using the transformed features.

When `cross_val_score` evaluates the Pipeline, each fold fits its own transformer using only that fold's training data. The validation fold is transformed with the parameters learned from the training portion.

In [ ]:
# Keep preprocessing and model training together as one estimator.
pipe_yeojohnson = Pipeline([
    ("power", PowerTransformer(method="yeo-johnson")),
    ("lr", LinearRegression())
])

# Evaluate the complete workflow over ten cross-validation folds.
yeojohnson_cv_scores = cross_val_score(
    pipe_yeojohnson,
    X,
    y,
    cv=10,
    scoring="r2"
)

print("Yeo-Johnson CV scores:", yeojohnson_cv_scores)
print("Mean Yeo-Johnson CV R²:", yeojohnson_cv_scores.mean())
print("Standard deviation:", yeojohnson_cv_scores.std())


## 11. Compare the fitted lambda values

Each feature receives its own λ because each feature can have a different distribution.

The table places the Box–Cox and Yeo–Johnson estimates side by side. Compare the estimates as learned parameters; do not assume that the method with a particular λ must have the better model score.

In [ ]:
# Compare the per-feature lambda estimates learned by the two transformations.
lambda_comparison = pd.DataFrame({
    "feature": X_train.columns,
    "box_cox_lambda": pt_boxcox.lambdas_,
    "yeo_johnson_lambda": pt_yeojohnson.lambdas_
})

display(lambda_comparison)


## 12. Compare the experiments

Collect the test-set scores and cross-validation results in one table. All three cross-validation experiments use ten folds, so their means and standard deviations are evaluated using the same fold count and Scikit-learn's default regression-folding strategy.

The test R² values come from the same held-out test set. Compare both the held-out scores and the cross-validation means; a small difference in one split may be due to sampling variation.

In [ ]:
# Summarize test scores and cross-validation statistics for all three workflows.
comparison = pd.DataFrame({
    "experiment": ["Baseline", "Box-Cox", "Yeo-Johnson"],
    "test_r2": [baseline_r2, boxcox_test_r2, yeojohnson_test_r2],
    "mean_cv_r2": [
        baseline_cv_scores.mean(),
        boxcox_cv_scores.mean(),
        yeojohnson_cv_scores.mean()
    ],
    "std_cv_r2": [
        baseline_cv_scores.std(),
        boxcox_cv_scores.std(),
        yeojohnson_cv_scores.std()
    ]
})

display(comparison)


## 13. Final observations

Use your actual outputs to complete these observations after running the notebook:

1. Which original features showed the strongest skew?
2. Which transformation made their distributions more symmetric?
3. Did the test R² improve, decrease, or remain similar after transformation?
4. Did the cross-validation results support the same conclusion?
5. Which method is suitable when a feature contains zero or negative values?

### Remember

- Box–Cox requires strictly positive inputs.
- Yeo–Johnson supports negative, zero, and positive inputs.
- `PowerTransformer` estimates one λ per feature during fitting.
- With the default `standardize=True`, Scikit-learn standardizes the power-transformed features after applying the power transformation.
- Fit preprocessing on training data only. A Pipeline helps enforce this rule during cross-validation.
- A more normal-looking feature distribution does not guarantee a better model. Compare against a baseline using an appropriate evaluation procedure.

For the mathematical derivations, deeper examples, limitations, and interview preparation, continue with **`notes.md`**.
